In [ ]:
import pandas as pd
from IPython.display import clear_output, display

def run_liquidation_model():
    # --- 1. Model State (Input Variables) ---
    state = {
        'initial_fcf': 100.0,
        'growth_rate': 0.05,
        'wacc': 0.10,
        'liquidation_value': 500.0,  # Immediate/Lifetime Exit Value
        'years': 5
    }

    while True:
        # --- 2. Calculations ---
        forecast = []
        pvs = []
        curr_fcf = state['initial_fcf']
        
        # Calculate Operating NPV
        for i in range(1, state['years'] + 1):
            curr_fcf *= (1 + state['growth_rate'])
            forecast.append(curr_fcf)
            pvs.append(curr_fcf / (1 + state['wacc'])**i)
        
        total_npv = sum(pvs)
        
        # Compare to Liquidation
        variance = total_npv - state['liquidation_value']
        best_path = "OPERATE (NPV)" if total_npv > state['liquidation_value'] else "LIQUIDATE"
        
        # --- 3. Display Interface ---
        clear_output(wait=True)
        print("====================================================")
        print("     NPV vs. LIFETIME LIQUIDATION SCENARIO MODEL    ")
        print("====================================================")
        print(f" INPUTS: FCF: ${state['initial_fcf']}M | Growth: {state['growth_rate']*100}% | WACC: {state['wacc']*100}%")
        print(f"         Liquidation/Exit Value: ${state['liquidation_value']}M")
        print("-" * 52)
        
        # Decision Summary
        summary_df = pd.DataFrame({
            "Valuation Path": ["Operating NPV", "Liquidation Value", "Variance (Value Added)"],
            "Amount ($M)": [f"${total_npv:,.2f}M", f"${state['liquidation_value']:,.2f}M", f"${variance:,.2f}M"]
        }).set_index("Valuation Path")
        
        display(summary_df)
        print(f"\n RECOMMENDED STRATEGY: {best_path}")
        print("-" * 52)
        
        # Year-by-Year Operating Forecast
        forecast_df = pd.DataFrame({
            "Year": [f"Year {i+1}" for i in range(state['years'])],
            "Forecasted FCF": [f"${x:,.2f}M" for x in forecast],
            "Present Value": [f"${x:,.2f}M" for x in pvs]
        }).set_index("Year")
        
        print("\n--- OPERATING CASH FLOW BREAKDOWN ---")
        display(forecast_df)

        # --- 4. User Input Interaction ---
        print("\n [1] Change Operating Variables (FCF/Growth/WACC)")
        print(" [2] Change Liquidation/Exit Value")
        print(" [R] Reset Defaults    [Q] Quit")
        
        choice = input("\nSelect an option to run a new scenario: ").strip().lower()

        if choice == 'q':
            break
        elif choice == 'r':
            state = {'initial_fcf': 100.0, 'growth_rate': 0.05, 'wacc': 0.10, 'liquidation_value': 500.0, 'years': 5}
        elif choice == '1':
            try:
                state['initial_fcf'] = float(input("Enter Initial FCF ($M): "))
                g = float(input("Enter Growth Rate (e.g., 0.05 for 5%): "))
                state['growth_rate'] = g if g < 1 else g/100
                w = float(input("Enter WACC (e.g., 0.10 for 10%): "))
                state['wacc'] = w if w < 1 else w/100
            except:
                input("Invalid input. Press Enter to retry...")
        elif choice == '2':
            try:
                state['liquidation_value'] = float(input("Enter Total Liquidation/Exit Value ($M): "))
            except:
                input("Invalid input. Press Enter to retry...")

run_liquidation_model()

     NPV vs. LIFETIME LIQUIDATION SCENARIO MODEL    
 INPUTS: FCF: $100.0M | Growth: 5.0% | WACC: 10.0%
         Liquidation/Exit Value: $500.0M
----------------------------------------------------


,Amount ($M)
Valuation Path,
Operating NPV,$435.81M
Liquidation Value,$500.00M
Variance (Value Added),$-64.19M



 RECOMMENDED STRATEGY: LIQUIDATE
----------------------------------------------------

--- OPERATING CASH FLOW BREAKDOWN ---


,Forecasted FCF,Present Value
Year,,
Year 1,$105.00M,$95.45M
Year 2,$110.25M,$91.12M
Year 3,$115.76M,$86.97M
Year 4,$121.55M,$83.02M
Year 5,$127.63M,$79.25M



 [1] Change Operating Variables (FCF/Growth/WACC)
 [2] Change Liquidation/Exit Value
 [R] Reset Defaults    [Q] Quit
